# Notebook 06b — Process-Aware Decision Optimization with Rescue Guardrails

## Goal

Notebook06 tested whether process-aware scoring can improve clone ranking by estimating expected post-optimization performance.

However, pure process-aware ranking can over-promote rescue clones and may diverge from the risk-controlled decision philosophy established in Notebook04.

The goal of Notebook06b is therefore:

> Test process-aware scoring inside the Notebook04-style decision framework.

---

## Key principle

This notebook keeps the core decision philosophy from Notebook04:

- Top-3 production candidates must remain protected
- Rescue clones are treated as optional process-optimization candidates
- Rescue candidates can enter Top-10, but only under controlled guardrails
- Process-aware scoring should improve decision quality, not simply increase rescue count

---

## What changes from Notebook06?

Notebook06:
- ranks all clones directly by process-aware score

Notebook06b:
- computes process-aware score
- applies pass / rescue / fail buckets
- applies rescue cap / guardrail
- compares baseline vs process-aware decision quality

---

## Main question

Does process-aware scoring improve Top-10 selection quality while preserving the risk-controlled decision logic from Notebook04?

In [1]:
# --------------------------------------------------
# Load prediction table from Notebook03
# --------------------------------------------------
import pandas as pd
import numpy as np
from pathlib import Path

scenario = "legacy"   # "legacy" or "optimized"
n_clones = 5000

PRED_PATH = Path("../data/synthetic/processed") / (
    f"predictions_03b_qp_{n_clones}_{scenario}.csv"
)

print("Loading:", PRED_PATH)

df = pd.read_csv(PRED_PATH)

print("Shape:", df.shape)
display(df.head())

required_cols = [
    "clone_id",
    "pred_late_qp",
    "pred_qp_drop",
    "pred_late_agg",
    "pred_stable_prob",
    "pred_rescue_score",
    "pred_rescue_label",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
]

missing = [c for c in required_cols if c not in df.columns]
print("Missing required columns:", missing)

assert len(missing) == 0, f"Missing required columns: {missing}"

Loading: ../data/synthetic/processed/predictions_03b_qp_5000_legacy.csv
Shape: (1000, 19)


,clone_id,pred_qp_drop,pred_late_qp,pred_late_agg,pred_stable_prob,pred_stable_label,true_qp_drop,true_late_qp,true_late_agg,true_stable_label,pred_super_prob,pred_aggr_prob,rescue_upside_qp,rescue_stability_band,rescue_quality_band,rescue_aggressive_penalty,rescue_not_already_pass,pred_rescue_score,pred_rescue_label
0,CLONE_1502,0.498165,2.673129e-08,8.435996,0.238692,0,0.206868,3.215605e-08,4.404966,1,0.001494,0.003219,0.000312,0.506117,0.981713,0.996769,0.710487,0.568590,0
1,CLONE_2587,0.373417,8.407121e-08,3.130765,0.464397,0,0.600740,5.393417e-08,3.662670,0,0.007656,0.000000,0.013124,0.921942,0.000000,1.000000,0.435364,0.416664,0
2,CLONE_2654,0.461606,2.770586e-08,7.582349,0.256485,0,0.156409,3.459109e-08,10.144181,1,0.001523,0.004162,0.000529,0.627981,0.737814,0.995823,0.688798,0.538817,0
3,CLONE_1056,0.348761,7.746555e-08,6.885601,0.507813,1,0.465880,6.392818e-08,2.383546,0,0.012355,0.000000,0.011648,0.991742,0.538743,1.000000,0.382442,0.594558,0
4,CLONE_0706,0.644735,1.744327e-07,8.275030,0.018172,0,0.678633,1.526521e-07,5.032915,0,0.004539,0.039329,0.033316,0.017551,0.935723,0.960529,0.979289,0.407241,0


Missing required columns: []


## Step 1 — Define utility and helper functions

We define the same utility logic used in earlier notebooks.

`true_util` is used only for offline evaluation.  
`baseline_score` is the original predicted utility before process optimization.

In [2]:
# --------------------------------------------------
# Step 1 — Utility and helper functions
# --------------------------------------------------

def z(s):
    s = pd.Series(s).astype(float)
    return (s - s.mean()) / (s.std(ddof=0) + 1e-9)

def z01(s):
    s = pd.Series(s).astype(float)
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

def make_true_utility(df):
    return (
        1.0 * z(df["true_late_qp"])
        - 1.0 * z(df["true_qp_drop"])
        - 0.2 * z(df["true_late_agg"])
    )

def make_pred_utility(df):
    return (
        1.0 * z(df["pred_late_qp"])
        - 1.0 * z(df["pred_qp_drop"])
        - 0.2 * z(df["pred_late_agg"])
    )

def topk_overlap(true_scores, pred_scores, k):
    true_top = set(pd.Series(true_scores).nlargest(k).index)
    pred_top = set(pd.Series(pred_scores).nlargest(k).index)
    return len(true_top & pred_top) / k

TOP_K = 10
TOP_STAGE2 = 3

df["true_util"] = make_true_utility(df)
df["baseline_score"] = make_pred_utility(df)

display(df[[
    "clone_id",
    "pred_late_qp",
    "pred_qp_drop",
    "pred_late_agg",
    "pred_rescue_score",
    "pred_rescue_label",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_util",
    "baseline_score",
]].head())

,clone_id,pred_late_qp,pred_qp_drop,pred_late_agg,pred_rescue_score,pred_rescue_label,true_late_qp,true_qp_drop,true_late_agg,true_util,baseline_score
0,CLONE_1502,2.673129e-08,0.498165,8.435996,0.568590,0,3.215605e-08,0.206868,4.404966,1.550478,-0.977650
1,CLONE_2587,8.407121e-08,0.373417,3.130765,0.416664,0,5.393417e-08,0.600740,3.662670,-0.804406,0.773624
2,CLONE_2654,2.770586e-08,0.461606,7.582349,0.538817,0,3.459109e-08,0.156409,10.144181,1.502466,-0.568919
3,CLONE_1056,7.746555e-08,0.348761,6.885601,0.594558,0,6.392818e-08,0.465880,2.383546,0.098570,0.649212
4,CLONE_0706,1.744327e-07,0.644735,8.275030,0.407241,0,1.526521e-07,0.678633,5.032915,-1.355847,-1.890491


## Step 2 — Recreate Notebook04-style decision buckets

We assign each clone to one of three groups:

- pass: strong predicted candidate
- rescue: borderline but process-optimization candidate
- fail: not selected

This keeps Notebook06b aligned with the decision philosophy of Notebook04.

In [3]:
# --------------------------------------------------
# Step 2 — Notebook04-style biosimilar bucket assignment
# --------------------------------------------------

BIO_AGG_PASS_MAX = 15.0
BIO_STABLE_PASS_MIN = 0.50

BIO_AGG_RESCUE_MAX = 18.0
BIO_STABLE_RESCUE_MIN = 0.25
BIO_RESCUE_SCORE_MIN = df["pred_rescue_score"].quantile(0.70)

work = df.copy()

work["bio_pass"] = (
    (work["pred_late_agg"] <= BIO_AGG_PASS_MAX) &
    (work["pred_stable_prob"] >= BIO_STABLE_PASS_MIN)
)

work["bio_rescue"] = (
    (~work["bio_pass"]) &
    (work["pred_rescue_score"] >= BIO_RESCUE_SCORE_MIN) &
    (work["pred_late_agg"] <= BIO_AGG_RESCUE_MAX) &
    (work["pred_stable_prob"] >= BIO_STABLE_RESCUE_MIN)
)

work["bio_bucket"] = "fail"
work.loc[work["bio_rescue"], "bio_bucket"] = "rescue"
work.loc[work["bio_pass"], "bio_bucket"] = "pass"

print("=== Bucket counts ===")
display(work["bio_bucket"].value_counts())

print("\n=== Bucket diagnostics ===")
display(work.groupby("bio_bucket")[[
    "pred_late_qp",
    "pred_qp_drop",
    "pred_late_agg",
    "pred_stable_prob",
    "pred_rescue_score",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_util",
]].mean())

=== Bucket counts ===


bio_bucket
fail      758
rescue    160
pass       82
Name: count, dtype: int64


=== Bucket diagnostics ===


,pred_late_qp,pred_qp_drop,pred_late_agg,pred_stable_prob,pred_rescue_score,true_late_qp,true_qp_drop,true_late_agg,true_util
bio_bucket,,,,,,,,,
fail,1.567428e-07,0.477066,5.778478,0.201773,0.404422,6.351721e-07,0.477744,5.722433,-0.130228
pass,2.935525e-07,0.328114,6.297133,0.574652,0.508005,4.325392e-07,0.310092,6.834508,0.805571
rescue,9.320008e-08,0.390032,8.285623,0.355348,0.618298,9.690855e-08,0.387139,8.472761,0.204099


## Step 3 — Safer process sensitivity model

Notebook06 used instability and aggregation risk as direct signs of process responsiveness.

Notebook06b uses a safer interpretation:

- high rescue score increases process sensitivity
- moderate instability may indicate room for process improvement
- extreme instability should not be rewarded too much
- aggregation risk may be improvable, but only within controlled limits

This avoids over-promoting risky clones.

In [4]:
# --------------------------------------------------
# Step 3 — Safer process sensitivity model
# --------------------------------------------------

def triangular_band_score(x, low, high, peak=None):
    x = pd.Series(x).astype(float)
    if peak is None:
        peak = (low + high) / 2

    score = pd.Series(np.zeros(len(x)), index=x.index)

    left = (x >= low) & (x <= peak)
    right = (x > peak) & (x <= high)

    score.loc[left] = (x.loc[left] - low) / (peak - low + 1e-12)
    score.loc[right] = (high - x.loc[right]) / (high - peak + 1e-12)

    return score.clip(0, 1)

# Rescue score remains the main signal
work["sens_rescue"] = z01(work["pred_rescue_score"])

# Moderate qP drop is considered optimizable.
# Very low drop has little room for improvement.
# Very high drop is too risky.
work["sens_stability_band"] = triangular_band_score(
    work["pred_qp_drop"],
    low=0.20,
    peak=0.35,
    high=0.60,
)

# Moderate aggregation is considered optimizable.
# Very high aggregation remains risky.
work["sens_agg_band"] = triangular_band_score(
    work["pred_late_agg"],
    low=5.0,
    peak=9.0,
    high=16.0,
)

work["process_sensitivity"] = (
    0.60 * work["sens_rescue"]
    + 0.25 * work["sens_stability_band"]
    + 0.15 * work["sens_agg_band"]
)

print("=== Process sensitivity summary ===")
display(work["process_sensitivity"].describe())

display(work[[
    "clone_id",
    "bio_bucket",
    "pred_rescue_score",
    "pred_qp_drop",
    "pred_late_agg",
    "sens_rescue",
    "sens_stability_band",
    "sens_agg_band",
    "process_sensitivity",
]].sort_values("process_sensitivity", ascending=False).head(15))

=== Process sensitivity summary ===


count    1000.000000
mean        0.474429
std         0.161631
min         0.000000
25%         0.375899
50%         0.469287
75%         0.587575
max         0.942601
Name: process_sensitivity, dtype: float64

,clone_id,bio_bucket,pred_rescue_score,pred_qp_drop,pred_late_agg,sens_rescue,sens_stability_band,sens_agg_band,process_sensitivity
180,CLONE_4625,pass,1.000000,0.322967,8.670837,1.000000,0.819778,0.917709,0.942601
269,CLONE_0150,rescue,0.714357,0.353381,8.895115,0.714357,0.986475,0.973779,0.821300
690,CLONE_4335,pass,0.717382,0.345661,8.935721,0.717382,0.971074,0.983930,0.820787
825,CLONE_1490,rescue,0.725636,0.351665,8.633797,0.725636,0.993340,0.908449,0.819984
182,CLONE_1942,rescue,0.733136,0.353795,8.453091,0.733136,0.984822,0.863273,0.815578
543,CLONE_4535,rescue,0.703903,0.361335,9.033933,0.703903,0.954658,0.995152,0.810279
569,CLONE_3867,rescue,0.697816,0.364354,9.033906,0.697816,0.942583,0.995156,0.803609
967,CLONE_1421,rescue,0.690321,0.346262,9.278052,0.690321,0.975083,0.960278,0.802005
743,CLONE_4054,rescue,0.688747,0.368455,8.998400,0.688747,0.926180,0.999600,0.794733
716,CLONE_3693,pass,0.706720,0.362765,8.547862,0.706720,0.948940,0.886966,0.794312


## Step 4 — Simulate expected process-adjusted phenotype

We convert process sensitivity into expected improvements.

Unlike Notebook06, this version uses controlled improvement ranges to avoid unrealistic promotion of risky clones.

In [5]:
# --------------------------------------------------
# Step 4 — Expected process-adjusted phenotype
# --------------------------------------------------

ALPHA_QP = 0.20       # productivity gain fraction
BETA_DROP = 0.10      # qP drop reduction
GAMMA_AGG = 0.20      # aggregation reduction fraction

work["expected_qp_gain_frac"] = ALPHA_QP * work["process_sensitivity"]
work["expected_drop_reduction"] = BETA_DROP * work["process_sensitivity"]
work["expected_agg_reduction_frac"] = GAMMA_AGG * work["process_sensitivity"]

work["opt_late_qp"] = (
    work["pred_late_qp"] * (1.0 + work["expected_qp_gain_frac"])
)

work["opt_qp_drop"] = (
    work["pred_qp_drop"] - work["expected_drop_reduction"]
).clip(lower=0.0)

work["opt_late_agg"] = (
    work["pred_late_agg"] * (1.0 - work["expected_agg_reduction_frac"])
).clip(lower=0.0)

display(work[[
    "clone_id",
    "bio_bucket",
    "process_sensitivity",
    "pred_late_qp",
    "opt_late_qp",
    "pred_qp_drop",
    "opt_qp_drop",
    "pred_late_agg",
    "opt_late_agg",
]].sort_values("process_sensitivity", ascending=False).head(15))

,clone_id,bio_bucket,process_sensitivity,pred_late_qp,opt_late_qp,pred_qp_drop,opt_qp_drop,pred_late_agg,opt_late_agg
180,CLONE_4625,pass,0.942601,4.500575e-06,5.349024e-06,0.322967,0.228707,8.670837,7.036209
269,CLONE_0150,rescue,0.821300,8.947176e-08,1.041684e-07,0.353381,0.271251,8.895115,7.434004
690,CLONE_4335,pass,0.820787,3.366004e-07,3.918559e-07,0.345661,0.263582,8.935721,7.468856
825,CLONE_1490,rescue,0.819984,7.682929e-08,8.942905e-08,0.351665,0.269667,8.633797,7.217882
182,CLONE_1942,rescue,0.815578,8.271182e-08,9.620341e-08,0.353795,0.272237,8.453091,7.074261
543,CLONE_4535,rescue,0.810279,1.715932e-07,1.994009e-07,0.361335,0.280308,9.033933,7.569931
569,CLONE_3867,rescue,0.803609,1.602156e-07,1.859657e-07,0.364354,0.283993,9.033906,7.581960
967,CLONE_1421,rescue,0.802005,8.803170e-08,1.021521e-07,0.346262,0.266062,9.278052,7.789843
743,CLONE_4054,rescue,0.794733,4.012741e-08,4.650552e-08,0.368455,0.288982,8.998400,7.568135
716,CLONE_3693,pass,0.794312,4.052925e-08,4.696783e-08,0.362765,0.283334,8.547862,7.189929


## Step 5 — Compute process-aware score with stability guardrail

The process-aware score is based on expected post-optimization performance.

However, process optimization should not be allowed to fully override risk.

Therefore, we also define an eligibility filter:

- pass clones remain eligible
- rescue clones are eligible only if optimized risk remains acceptable
- fail clones are excluded from decision-stage ranking

In [6]:
# --------------------------------------------------
# Step 5 — Process-aware score + eligibility guardrail
# --------------------------------------------------

work["process_aware_score"] = (
    1.0 * z(work["opt_late_qp"])
    - 1.0 * z(work["opt_qp_drop"])
    - 0.2 * z(work["opt_late_agg"])
)

# Eligibility guardrail
MAX_OPT_DROP = 0.55
MAX_OPT_AGG = 16.0

work["eligible_for_process_decision"] = (
    (work["bio_bucket"].isin(["pass", "rescue"])) &
    (work["opt_qp_drop"] <= MAX_OPT_DROP) &
    (work["opt_late_agg"] <= MAX_OPT_AGG)
)

print("=== Eligibility counts ===")
display(work["eligible_for_process_decision"].value_counts())

print("\n=== Eligible bucket counts ===")
display(work[work["eligible_for_process_decision"]]["bio_bucket"].value_counts())

display(work[[
    "clone_id",
    "bio_bucket",
    "eligible_for_process_decision",
    "baseline_score",
    "process_aware_score",
    "process_sensitivity",
    "opt_late_qp",
    "opt_qp_drop",
    "opt_late_agg",
    "true_util",
]].sort_values("process_aware_score", ascending=False).head(15))

=== Eligibility counts ===


eligible_for_process_decision
False    758
True     242
Name: count, dtype: int64


=== Eligible bucket counts ===


bio_bucket
rescue    160
pass       82
Name: count, dtype: int64

,clone_id,bio_bucket,eligible_for_process_decision,baseline_score,process_aware_score,process_sensitivity,opt_late_qp,opt_qp_drop,opt_late_agg,true_util
180,CLONE_4625,pass,True,12.718873,13.851003,0.942601,5.349024e-06,0.228707,7.036209,3.040391
621,CLONE_3254,pass,True,12.974673,13.185438,0.675180,4.912918e-06,0.178719,7.227716,2.451734
505,CLONE_0048,fail,False,9.293532,9.431781,0.569914,4.578249e-06,0.507639,9.468847,1.893128
730,CLONE_4878,fail,False,7.933686,8.226819,0.629453,3.953383e-06,0.482602,8.445963,0.525635
524,CLONE_2171,fail,False,7.964768,8.192512,0.557017,4.495513e-06,0.657783,7.519941,-1.862933
314,CLONE_1619,pass,True,8.901264,8.167500,0.275294,2.775428e-06,0.208661,3.171964,1.978856
986,CLONE_2776,fail,False,7.992076,7.784792,0.419982,4.065592e-06,0.594883,6.379565,34.138263
63,CLONE_3863,fail,False,5.105205,5.117253,0.454569,3.330291e-06,0.693011,7.105434,-1.962704
23,CLONE_0643,fail,False,4.466508,4.618398,0.597304,2.179790e-06,0.422463,6.501972,-0.332074
183,CLONE_0101,fail,False,3.104626,3.388024,0.720871,1.390905e-06,0.337931,6.631902,-0.266375


## Step 6 — Apply Notebook04-style rescue guardrail

This is the key difference from Notebook06.

We do not simply take the Top-10 by process-aware score.

Instead:

- Top-3 remains pure process-aware ranking among eligible clones
- Top-10 is process-aware ranking
- If no rescue clone appears in Top-10, one best rescue clone may be added
- Rescue inclusion is controlled and does not distort Top-3

In [7]:
# --------------------------------------------------
# Step 6 — Notebook04-style rescue guardrail
# --------------------------------------------------

def apply_rescue_guardrail(
    stage_df,
    bucket_col,
    score_col,
    top_n=10,
    top_stage2=3,
    min_rescue_top10=1,
    max_rescue_top10=2,
):
    ranked = stage_df.sort_values(score_col, ascending=False).copy()

    # Top3 is pure ranking
    top3 = ranked.head(top_stage2).copy()

    # Top10 starts as pure ranking
    top10 = ranked.head(top_n).copy()

    protected_ids = set(top3["clone_id"])
    top10_ids = set(top10["clone_id"])

    current_rescue_n = int((top10[bucket_col] == "rescue").sum())

    # If too many rescue clones, cap them by replacing lowest rescue with next-best pass clones
    if current_rescue_n > max_rescue_top10:
        rescue_keep = (
            top10[top10[bucket_col] == "rescue"]
            .sort_values(score_col, ascending=False)
            .head(max_rescue_top10)
        )

        pass_keep = top10[top10[bucket_col] != "rescue"]

        needed = top_n - len(rescue_keep) - len(pass_keep)

        pass_pool = ranked[
            (ranked[bucket_col] == "pass") &
            (~ranked["clone_id"].isin(set(pass_keep["clone_id"]))) &
            (~ranked["clone_id"].isin(set(rescue_keep["clone_id"])))
        ].head(max(0, needed))

        top10 = pd.concat([pass_keep, rescue_keep, pass_pool], ignore_index=True)
        top10 = top10.sort_values(score_col, ascending=False).head(top_n)
        current_rescue_n = int((top10[bucket_col] == "rescue").sum())

    # If no rescue clone appears, add one best rescue candidate outside protected Top3
    if current_rescue_n < min_rescue_top10:
        rescue_pool = ranked[
            (ranked[bucket_col] == "rescue") &
            (~ranked["clone_id"].isin(protected_ids)) &
            (~ranked["clone_id"].isin(set(top10["clone_id"])))
        ].copy()

        if len(rescue_pool) > 0:
            rescue_add = rescue_pool.head(min_rescue_top10 - current_rescue_n)

            replace_candidates = (
                top10[
                    (~top10["clone_id"].isin(protected_ids)) &
                    (top10[bucket_col] != "rescue")
                ]
                .sort_values(score_col, ascending=True)
                .head(len(rescue_add))
            )

            top10 = pd.concat(
                [
                    top10[~top10["clone_id"].isin(replace_candidates["clone_id"])],
                    rescue_add,
                ],
                ignore_index=True,
            ).sort_values(score_col, ascending=False).head(top_n)

    return top10.reset_index(drop=True), top3.reset_index(drop=True)

eligible = work[work["eligible_for_process_decision"]].copy()

process_top10, process_top3 = apply_rescue_guardrail(
    stage_df=eligible,
    bucket_col="bio_bucket",
    score_col="process_aware_score",
    top_n=TOP_K,
    top_stage2=TOP_STAGE2,
    min_rescue_top10=1,
    max_rescue_top10=2,
)

print("=== Process-aware guarded selection ===")
print("Top10 rescue count:", int((process_top10["bio_bucket"] == "rescue").sum()))
print("Top3 rescue count:", int((process_top3["bio_bucket"] == "rescue").sum()))

display(process_top10[[
    "clone_id",
    "bio_bucket",
    "process_aware_score",
    "baseline_score",
    "process_sensitivity",
    "pred_rescue_score",
    "opt_late_qp",
    "opt_qp_drop",
    "opt_late_agg",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_util",
]])

=== Process-aware guarded selection ===
Top10 rescue count: 2
Top3 rescue count: 0


,clone_id,bio_bucket,process_aware_score,baseline_score,process_sensitivity,pred_rescue_score,opt_late_qp,opt_qp_drop,opt_late_agg,true_late_qp,true_qp_drop,true_late_agg,true_util
0,CLONE_4625,pass,13.851003,12.718873,0.942601,1.000000,5.349024e-06,0.228707,7.036209,1.238325e-05,0.084353,10.114743,3.040391
1,CLONE_3254,pass,13.185438,12.974673,0.675180,0.787107,4.912918e-06,0.178719,7.227716,7.103148e-06,0.089331,11.556387,2.451734
2,CLONE_1619,pass,8.167500,8.901264,0.275294,0.358295,2.775428e-06,0.208661,3.171964,2.544839e-06,0.164530,5.259631,1.978856
3,CLONE_2759,pass,3.042048,3.183487,0.522149,0.481011,9.799322e-07,0.221138,9.678081,1.094417e-06,0.220844,16.093701,0.834379
4,CLONE_0531,rescue,2.313373,2.090859,0.730366,0.637943,8.787162e-07,0.280085,9.649084,7.255973e-07,0.400696,7.821784,0.217604
5,CLONE_3581,pass,2.058350,2.434677,0.264514,0.251555,3.760899e-07,0.241697,3.319856,3.901131e-07,0.252349,5.063234,1.263958
6,CLONE_3393,rescue,1.943382,1.624424,0.787522,0.722734,7.435935e-07,0.312275,7.468078,8.308232e-07,0.314570,3.423687,1.025221
7,CLONE_1039,pass,1.899008,2.264493,0.257586,0.245423,2.821471e-07,0.240441,2.751018,2.564530e-07,0.324499,0.127177,1.118160
8,CLONE_3969,pass,1.869120,2.084520,0.408819,0.376402,4.206794e-07,0.254870,5.163954,4.846742e-07,0.168596,4.382455,1.825537
9,CLONE_2358,pass,1.859624,1.977395,0.469408,0.418395,4.842293e-07,0.284082,4.410059,5.671183e-07,0.204971,4.884648,1.579792


## Step 7 — Build baseline guarded selection

For a fair comparison, we apply the same Notebook04-style guardrail to baseline scoring.

This allows us to compare:

- baseline guarded decision
- process-aware guarded decision

under the same decision framework.

In [8]:
# --------------------------------------------------
# Step 7 — Baseline guarded selection
# --------------------------------------------------

baseline_eligible = work[
    (work["bio_bucket"].isin(["pass", "rescue"])) &
    (work["pred_qp_drop"] <= MAX_OPT_DROP) &
    (work["pred_late_agg"] <= MAX_OPT_AGG)
].copy()

baseline_top10, baseline_top3 = apply_rescue_guardrail(
    stage_df=baseline_eligible,
    bucket_col="bio_bucket",
    score_col="baseline_score",
    top_n=TOP_K,
    top_stage2=TOP_STAGE2,
    min_rescue_top10=1,
    max_rescue_top10=2,
)

print("=== Baseline guarded selection ===")
print("Top10 rescue count:", int((baseline_top10["bio_bucket"] == "rescue").sum()))
print("Top3 rescue count:", int((baseline_top3["bio_bucket"] == "rescue").sum()))

display(baseline_top10[[
    "clone_id",
    "bio_bucket",
    "baseline_score",
    "process_aware_score",
    "pred_rescue_score",
    "pred_late_qp",
    "pred_qp_drop",
    "pred_late_agg",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_util",
]])

=== Baseline guarded selection ===
Top10 rescue count: 1
Top3 rescue count: 0


,clone_id,bio_bucket,baseline_score,process_aware_score,pred_rescue_score,pred_late_qp,pred_qp_drop,pred_late_agg,true_late_qp,true_qp_drop,true_late_agg,true_util
0,CLONE_3254,pass,12.974673,13.185438,0.787107,4.328425e-06,0.246238,8.356089,7.103148e-06,0.089331,11.556387,2.451734
1,CLONE_4625,pass,12.718873,13.851003,1.000000,4.500575e-06,0.322967,8.670837,1.238325e-05,0.084353,10.114743,3.040391
2,CLONE_1619,pass,8.901264,8.167500,0.358295,2.630590e-06,0.236190,3.356785,2.544839e-06,0.164530,5.259631,1.978856
3,CLONE_2759,pass,3.183487,3.042048,0.481011,8.872743e-07,0.273353,10.806613,1.094417e-06,0.220844,16.093701,0.834379
4,CLONE_3581,pass,2.434677,2.058350,0.251555,3.571934e-07,0.268148,3.505295,3.901131e-07,0.252349,5.063234,1.263958
5,CLONE_1039,pass,2.264493,1.899008,0.245423,2.683238e-07,0.266199,2.900440,2.564530e-07,0.324499,0.127177,1.118160
6,CLONE_3822,pass,2.107874,1.826439,0.301876,2.854552e-07,0.287347,3.031813,3.452741e-07,0.156283,5.447983,1.822173
7,CLONE_0531,rescue,2.090859,2.313373,0.637943,7.667192e-07,0.353122,11.299661,7.255973e-07,0.400696,7.821784,0.217604
8,CLONE_3969,pass,2.084520,1.869120,0.376402,3.888828e-07,0.295752,5.623775,4.846742e-07,0.168596,4.382455,1.825537
9,CLONE_2378,pass,2.016318,1.815483,0.363440,3.525990e-07,0.308969,3.917373,3.382243e-07,0.322691,5.641511,0.794344


## Step 8 — Compare baseline vs process-aware guarded decisions

Now we compare the two final Top-10 sets under the same decision rule.

This tells us whether process-aware scoring improves selection without changing the overall decision philosophy.

In [9]:
# --------------------------------------------------
# Step 8 — Decision comparison
# --------------------------------------------------

baseline_ids = set(baseline_top10["clone_id"])
process_ids = set(process_top10["clone_id"])

decision_overlap = len(baseline_ids & process_ids)

print("=== Decision comparison ===")
print(f"Top10 overlap: {decision_overlap}/{TOP_K}")
print("Baseline-only clones:", sorted(list(baseline_ids - process_ids)))
print("Process-aware-only clones:", sorted(list(process_ids - baseline_ids)))

comparison_summary = pd.DataFrame({
    "baseline_guarded_top10": baseline_top10[[
        "true_late_qp",
        "true_qp_drop",
        "true_late_agg",
        "true_util",
    ]].mean(),
    "process_guarded_top10": process_top10[[
        "true_late_qp",
        "true_qp_drop",
        "true_late_agg",
        "true_util",
    ]].mean(),
})

comparison_summary["direction"] = [
    "higher is better",
    "lower is better",
    "lower is better",
    "higher is better",
]

display(comparison_summary)

=== Decision comparison ===
Top10 overlap: 8/10
Baseline-only clones: ['CLONE_2378', 'CLONE_3822']
Process-aware-only clones: ['CLONE_2358', 'CLONE_3393']


,baseline_guarded_top10,process_guarded_top10,direction
true_late_qp,0.000003,0.000003,higher is better
true_qp_drop,0.218417,0.222474,lower is better
true_late_agg,7.150861,6.872745,lower is better
true_util,1.534714,1.533563,higher is better


## Step 9 — Utility overlap evaluation

This is the main validation step.

We compare:

- true Top-10 by `true_util`
- baseline guarded Top-10
- process-aware guarded Top-10

If process-aware guarded overlap is higher, then process-aware scoring improved true Top-10 recovery inside the decision framework.

In [10]:
# --------------------------------------------------
# Step 9 — Utility overlap evaluation
# --------------------------------------------------

true_top_ids = set(work.sort_values("true_util", ascending=False).head(TOP_K)["clone_id"])

baseline_overlap = len(true_top_ids & set(baseline_top10["clone_id"])) / TOP_K
process_overlap = len(true_top_ids & set(process_top10["clone_id"])) / TOP_K

print("=== Utility overlap ===")
print(f"Baseline guarded utility overlap:      {baseline_overlap:.3f}")
print(f"Process-aware guarded utility overlap: {process_overlap:.3f}")

if process_overlap > baseline_overlap:
    print("\nInterpretation: Process-aware guarded scoring improved true Top-K recovery.")
elif process_overlap == baseline_overlap:
    print("\nInterpretation: Process-aware guarded scoring preserved true Top-K recovery.")
else:
    print("\nInterpretation: Process-aware guarded scoring reduced true Top-K recovery.")

=== Utility overlap ===
Baseline guarded utility overlap:      0.200
Process-aware guarded utility overlap: 0.200

Interpretation: Process-aware guarded scoring preserved true Top-K recovery.


## Final Interpretation

Notebook06b tests process-aware scoring under the same risk-controlled decision philosophy as Notebook04.

### What changed from Notebook06?

Notebook06 used direct process-aware ranking.

Notebook06b uses:

- Notebook04-style pass / rescue / fail buckets
- eligibility guardrails
- Top-3 protection
- rescue minimum / maximum guardrails
- fair comparison between baseline guarded and process-aware guarded decisions

---

## How to interpret the results

### Good outcome

- Top10 changes moderately
- rescue count remains controlled
- Top3 rescue count remains 0
- utility overlap improves or is preserved
- true utility increases or remains stable

This means process-aware scoring can be safely integrated into the decision engine.

---

### Risky outcome

- rescue count becomes too high
- Top3 includes rescue clones
- true qP drop worsens
- utility overlap decreases

This means process-aware scoring is too aggressive or misaligned with true biology.

---

## Project meaning

Notebook06b connects:

Prediction → Decision → Process-aware Optimization

while preserving the core production logic:

> Rescue clones are process-optimization opportunities, not automatic production candidates.

---

## Next step

If Notebook06b is stable, Notebook07 can introduce explicit media / nutrient optimization:

- nutrient-limited vs nutrient-rich conditions
- clone-specific media sensitivity
- productivity / stability / quality trade-offs
- clone × process pairing